# 08 经营 KPI 分析

核心经营指标计算与可视化：
- GMV 月度趋势与环比增长
- 品类收入与毛利贡献
- 区域 GMV 分布
- 支付方式分析与分期行为

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from utils.db_connector import get_engine
engine = get_engine()

## 8.1 加载订单数据

In [ ]:
# 加载订单数据（多表关联）
query = """
SELECT 
    o.order_id, o.customer_id, o.order_purchase_timestamp, o.order_status,
    i.product_id, i.seller_id, i.price, i.freight_value,
    p.product_category_name, c.customer_state,
    pay.payment_type, pay.payment_installments, pay.payment_value
FROM olist_orders o
LEFT JOIN olist_order_items i ON o.order_id = i.order_id
LEFT JOIN olist_products p ON i.product_id = p.product_id
LEFT JOIN olist_customers c ON o.customer_id = c.customer_id
LEFT JOIN olist_order_payments pay ON o.order_id = pay.order_id
WHERE o.order_status = 'delivered'
"""

with engine.connect() as conn:
    df = pd.read_sql_query(query, conn)

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['year_month'] = df['order_purchase_timestamp'].dt.to_period('M')

# 毛利率假设
margin_rates = {
    '电脑配件': 0.12, '数码电子': 0.15, '手机通讯': 0.30,
    '美妆个护': 0.50, '运动户外': 0.40, '家居装饰': 0.35,
    '家居日用': 0.42, '汽车用品': 0.25, '玩具': 0.40,
    '创意潮品': 0.35, '家用电器': 0.30
}
df['margin_rate'] = df['product_category_name'].map(lambda x: margin_rates.get(str(x), 0.30))
df['gross_profit'] = df['price'] * df['margin_rate']

print(f"已加载订单数: {df['order_id'].nunique():,}")
print(f"时间范围: {df['order_purchase_timestamp'].min()} ~ {df['order_purchase_timestamp'].max()}")
df.head()

## 8.2 月度 KPI 计算

In [ ]:
# 月度 KPI
monthly = df.groupby('year_month').agg(
    gmv=('price', 'sum'),
    freight_revenue=('freight_value', 'sum'),
    order_count=('order_id', 'nunique'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

monthly['aov'] = monthly['gmv'] / monthly['order_count']
monthly['gross_margin'] = monthly['gross_profit'] / monthly['gmv']
monthly['gmv_growth'] = monthly['gmv'].pct_change()
monthly['month'] = monthly['year_month'].astype(str)

print(f"总 GMV: ¥{monthly['gmv'].sum():,.2f}")
print(f"平均月 GMV: ¥{monthly['gmv'].mean():,.2f}")
print(f"平均客单价: ¥{monthly['aov'].mean():.2f}")
print(f"平均毛利率: {monthly['gross_margin'].mean():.2%}")
monthly

## 8.3 GMV 月度趋势

In [ ]:
# GMV + 环比增长率双轴图
fig, ax1 = plt.subplots(figsize=(14, 6))

x = range(len(monthly))
ax1.bar(x, monthly['gmv'] / 10000, color='#3498db', alpha=0.7, label='GMV(万)')
ax1.set_xlabel('月份')
ax1.set_ylabel('GMV (万)', color='#3498db')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax1.set_xticks(x)
ax1.set_xticklabels(monthly['month'], rotation=45, ha='right', fontsize=8)

ax2 = ax1.twinx()
growth_pct = monthly['gmv_growth'] * 100
ax2.plot(x, growth_pct, color='#e74c3c', marker='o', linewidth=2, label='环比增长率(%)')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('环比增长率 (%)', color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.title('月度 GMV 趋势与环比增长率')
plt.tight_layout()
plt.show()

## 8.4 品类表现分析

In [ ]:
# 品类分析
category = df.groupby('product_category_name').agg(
    revenue=('price', 'sum'),
    order_count=('order_id', 'nunique'),
    gross_profit=('gross_profit', 'sum'),
    avg_price=('price', 'mean')
).reset_index()
category['margin_rate'] = category['gross_profit'] / category['revenue']
category = category.sort_values('revenue', ascending=False)

print(f"品类数: {len(category)}")
print("\nTop 10 品类收入:")
category.head(10)[['product_category_name', 'revenue', 'order_count', 'margin_rate']].round(2)

In [ ]:
# 品类收入 & 毛利可视化
top10 = category.head(10)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(len(top10)), top10['revenue'] / 10000, color='#3498db', alpha=0.8)
axes[0].set_yticks(range(len(top10)))
axes[0].set_yticklabels(top10['product_category_name'], fontsize=9)
axes[0].set_xlabel('收入 (万)')
axes[0].set_title('品类收入 Top 10')
axes[0].invert_yaxis()

top10_margin = category.nlargest(10, 'gross_profit')
axes[1].barh(range(len(top10_margin)), top10_margin['gross_profit'] / 10000, color='#2ecc71', alpha=0.8)
axes[1].set_yticks(range(len(top10_margin)))
axes[1].set_yticklabels(top10_margin['product_category_name'], fontsize=9)
axes[1].set_xlabel('毛利 (万)')
axes[1].set_title('品类毛利贡献 Top 10')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 8.5 区域分析

In [ ]:
# 地区 GMV 分析
state = df.groupby('customer_state').agg(
    gmv=('price', 'sum'),
    order_count=('order_id', 'nunique'),
    customer_count=('customer_id', 'nunique'),
    avg_order_value=('price', 'mean')
).reset_index().sort_values('gmv', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(state)), state['gmv'] / 10000, color='#9b59b6', alpha=0.7)
ax.set_xticks(range(len(state)))
ax.set_xticklabels(state['customer_state'], rotation=45, ha='right')
ax.set_xlabel('省份/州')
ax.set_ylabel('GMV (万)')
ax.set_title('各省份 GMV 分布（降序）')
plt.tight_layout()
plt.show()

print("Top 5 省份:")
state.head(5)[['customer_state', 'gmv', 'order_count', 'customer_count']]

## 8.6 支付方式分析

In [ ]:
# 支付方式分析
payment_df = df.drop_duplicates(subset=['order_id', 'payment_type'])
payment = payment_df.groupby('payment_type').agg(
    transaction_count=('order_id', 'nunique'),
    total_value=('payment_value', 'sum'),
    avg_installments=('payment_installments', 'mean')
).reset_index()
payment['share'] = payment['transaction_count'] / payment['transaction_count'].sum()
payment = payment.sort_values('transaction_count', ascending=False)

# 饼图
fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
ax.pie(payment['transaction_count'], labels=payment['payment_type'],
       autopct='%1.1f%%', colors=colors[:len(payment)], startangle=90)
ax.set_title('支付方式占比')
plt.tight_layout()
plt.show()

payment

## 8.7 分期付款行为

In [ ]:
# 分期付款分析
installments = df.drop_duplicates(subset=['order_id', 'payment_installments'])
inst_dist = installments['payment_installments'].value_counts().sort_index()
inst_dist = inst_dist[inst_dist.index <= 12]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(inst_dist.index, inst_dist.values, color='#f39c12', alpha=0.8)
axes[0].set_xlabel('分期期数')
axes[0].set_ylabel('订单数')
axes[0].set_title('分期期数分布')

# 月度平均分期趋势
installments['year_month'] = installments['order_purchase_timestamp'].dt.to_period('M')
monthly_inst = installments.groupby('year_month')['payment_installments'].mean()
axes[1].plot(range(len(monthly_inst)), monthly_inst.values, marker='o', color='#e67e22', linewidth=2)
axes[1].set_xlabel('月份')
axes[1].set_ylabel('平均分期期数')
axes[1].set_title('平均分期期数月度趋势')

plt.tight_layout()
plt.show()

## 小结

- GMV 整体呈增长趋势，2018年上半年增速最快
- 电子产品和家居类目是主力收入品类
- SP、RJ 等州贡献了绝大部分 GMV
- 信用卡为主要支付方式，分期购买行为常见
- 高毛利品类（美妆、运动）值得加大运营投入